# Ink Detection Challenge - Extreme Inference Notebook

This notebook implements the **Extreme Inference** pipeline (8-way TTA + 75% Overlap) for the Vesuvius Ink Detection Challenge. 

**Instructions:**
1. Connect to a T4 GPU (Runtime -> Change runtime type).
2. Mount your Google Drive to access the fragments and model weights.
3. Set the paths in the configuration cell.

In [8]:
# 1. Environment Setup
!pip install -q timm scikit-image tqdm pillow

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!nvidia-smi

Sat May 30 11:53:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             28W /   70W |    1107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Model Architecture (V2)

In [10]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout_prob=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_prob)
        )

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class InkDetectionNetV2(nn.Module):
    def __init__(self, encoder_name='tf_efficientnetv2_l', pretrained=True, input_channels=30, dropout_prob=0.3):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained, in_chans=input_channels, features_only=True, out_indices=(0, 1, 2, 3, 4))
        encoder_channels = self.encoder.feature_info.channels()
        self.dec1 = DecoderBlock(encoder_channels[4], encoder_channels[3], 512, dropout_prob)
        self.dec2 = DecoderBlock(512, encoder_channels[2], 256, dropout_prob)
        self.dec3 = DecoderBlock(256, encoder_channels[1], 128, dropout_prob)
        self.dec4 = DecoderBlock(128, encoder_channels[0], 64, dropout_prob)
        self.dec5 = DecoderBlock(64, 0, 32, dropout_prob)
        self.final_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        features = self.encoder(x)
        x = self.dec1(features[4], features[3])
        x = self.dec2(x, features[2])
        x = self.dec3(x, features[1])
        x = self.dec4(x, features[0])
        x = self.dec5(x)
        return self.final_conv(x)

    def predict(self, x):
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)
            return torch.sigmoid(logits)

## 3. Data Management

In [11]:
class InferenceDataManager:
    def __init__(self, data_dir, patch_size=256, stride=128, slices=(20, 45)):
        self.data_dir = Path(data_dir)
        self.patch_size = patch_size
        self.stride = stride
        self.slices = slices
        self.positions = []
        self.volume = None
        self.image_shape = (0, 0)

    def set_fragment(self, fragment_id):
        volume_path = self.data_dir / fragment_id / "surface_volume_batch.npy"
        if not volume_path.exists():
            raise FileNotFoundError(f"Volume not found at {volume_path}")
        self.volume = np.load(volume_path, mmap_mode='r')
        h, w = self.volume.shape[1], self.volume.shape[2]
        self.image_shape = (h, w)
        self.positions = []
        for y in range(0, h - self.patch_size + 1, self.stride):
            for x in range(0, w - self.patch_size + 1, self.stride):
                self.positions.append((y, x))
        # Edges
        if (h - self.patch_size) % self.stride != 0:
            for x in range(0, w - self.patch_size + 1, self.stride): self.positions.append((h - self.patch_size, x))
        if (w - self.patch_size) % self.stride != 0:
            for y in range(0, h - self.patch_size + 1, self.stride): self.positions.append((y, w - self.patch_size))
        self.positions.append((h - self.patch_size, w - self.patch_size))
        self.positions = sorted(list(set(self.positions)))

    def get_patch(self, patch_idx):
        y, x = self.positions[patch_idx]
        z1, z2 = self.slices
        patch_3d = self.volume[z1:z2, y:y+self.patch_size, x:x+self.patch_size]
        patch_3d = patch_3d.astype(np.float32) / 65535.0
        return patch_3d, (y, x)

    def __len__(self): return len(self.positions)

## 4. Extreme Inference Logic

In [12]:
def get_hanning_window(size):
    win = np.hanning(size)
    return np.outer(win, win).astype(np.float32)

def predict_tta(model, x, device):
    total_prob = torch.zeros((x.shape[0], 1, x.shape[2], x.shape[3]), device=device)
    with torch.no_grad():
        total_prob += model.predict(x)
        total_prob += torch.flip(model.predict(torch.flip(x, dims=[-1])), dims=[-1])
        total_prob += torch.flip(model.predict(torch.flip(x, dims=[-2])), dims=[-2])
        total_prob += torch.flip(model.predict(torch.flip(x, dims=[-1, -2])), dims=[-1, -2])
        x_t = x.transpose(-1, -2)
        total_prob += model.predict(x_t).transpose(-1, -2)
        total_prob += torch.rot90(model.predict(torch.rot90(x, k=1, dims=[-2, -1])), k=-1, dims=[-2, -1])
        total_prob += torch.rot90(model.predict(torch.rot90(x, k=3, dims=[-2, -1])), k=-3, dims=[-2, -1])
        total_prob += torch.flip(model.predict(torch.flip(x_t, dims=[-1])), dims=[-1]).transpose(-1, -2)
    return total_prob / 8.0

## 5. Main Inference Execution

In [14]:
# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/InkDetection/test" # ADJUST THIS
CHECKPOINT_PATH = "/content/drive/MyDrive/InkDetection/best_ink_model.pth" # ADJUST THIS
OUTPUT_DIR = "./results"
PATCH_SIZE = 256
STRIDE = 64 # 75% overlap
THRESHOLD = 0.65

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# 1. Load Model
model = InkDetectionNetV2(input_channels=30, pretrained=False).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Model Loaded from {CHECKPOINT_PATH}")

# 2. Initialize
manager = InferenceDataManager(DATA_DIR, patch_size=PATCH_SIZE, stride=STRIDE, slices=(15, 45))
hanning_win = get_hanning_window(PATCH_SIZE)
fragment_ids = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]

# 3. Run Loop
for fid in fragment_ids:
    print(f"\nProcessing {fid}...")
    manager.set_fragment(fid)
    full_prob_map = np.zeros(manager.image_shape, dtype=np.float32)
    weight_map = np.zeros(manager.image_shape, dtype=np.float32)

    for i in tqdm(range(len(manager)), desc=f"Inference {fid}"):
        patch_3d, (y, x) = manager.get_patch(i)
        patch_tensor = torch.from_numpy(patch_3d).unsqueeze(0).to(device)
        prob = predict_tta(model, patch_tensor, device).squeeze().cpu().numpy()
        full_prob_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += (prob * hanning_win)
        weight_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += hanning_win

    final_prob_map = full_prob_map / (weight_map + 1e-7)
    binary_mask = (final_prob_map >= THRESHOLD).astype(np.uint8)
    
    # Post-processing
    from skimage import morphology
    labeled = morphology.label(binary_mask)
    clean = morphology.remove_small_objects(labeled, min_size=512)
    binary_mask = (clean > 0).astype(np.uint8) * 255
    
    # Save
    Image.fromarray(binary_mask).save(f"{OUTPUT_DIR}/{fid}_prediction.png")
    print(f"Saved {fid} results.")

Model Loaded from /content/drive/MyDrive/InkDetection/best_ink_model.pth

Processing p_3homka84a6c1...


Inference p_3homka84a6c1:   0%|          | 0/841 [00:00<?, ?it/s]

Saved p_3homka84a6c1 results.
